# Segmentation
## Goal
### 멀티모달에 적용하기 위해 Segmentation을 통해 이미지의 수치를 추출하자!

## 데이터셋 가져오기

In [1]:
import os 

dataset_name = "20230605_ch1_crop"
origin_path = f"./dataset/image/{dataset_name}/test/Image/"

imgs=os.listdir(origin_path)

seg_type = 'cloud' #'soiling'


try : 
    imgs.remove('.ipynb_checkpoints') 
except :
    print("폴더 내에 .ipynb_checkpoints 없음")

imgs

폴더 내에 .ipynb_checkpoints 없음


['panel1_NVR_ch1_jpg_20230322164400.jpg',
 'panel1_NVR_ch1_jpg_20230322164800.jpg',
 'panel1_NVR_ch1_jpg_20230322165200.jpg',
 'panel1_NVR_ch1_jpg_20230322165600.jpg',
 'panel1_NVR_ch1_jpg_20230322170200.jpg',
 'panel1_NVR_ch1_jpg_20230329091700.jpg',
 'panel1_NVR_ch1_jpg_20230329092100.jpg',
 'panel1_NVR_ch1_jpg_20230329092500.jpg',
 'panel1_NVR_ch1_jpg_20230329092900.jpg',
 'panel1_NVR_ch1_jpg_20230329093300.jpg',
 'panel1_NVR_ch1_jpg_20230329093700.jpg',
 'panel1_NVR_ch1_jpg_20230329094200.jpg',
 'panel1_NVR_ch1_jpg_20230329141400.jpg',
 'panel1_NVR_ch1_jpg_20230329141800.jpg',
 'panel1_NVR_ch1_jpg_20230329142200.jpg',
 'panel1_NVR_ch1_jpg_20230329142600.jpg',
 'panel1_NVR_ch1_jpg_20230329143000.jpg',
 'panel1_NVR_ch1_jpg_20230329143400.jpg',
 'panel1_NVR_ch1_jpg_20230329144200.jpg',
 'panel1_NVR_ch1_jpg_20230329143800.jpg',
 'panel1_NVR_ch1_jpg_20230329144600.jpg',
 'panel1_NVR_ch1_jpg_20230329145000.jpg',
 'panel1_NVR_ch1_jpg_20230329145400.jpg',
 'panel1_NVR_ch1_jpg_2023032914580

## Model
### interface에서 썼던 SegmentationModels Class
- 출처 cctv > segmentationCam.py
- type만 추가

In [2]:
class SegmentationModels(object):
    def __init__(self, modelName, result_type):
        print("SegmentationCam.py __init__ 들어옴")

        self.modelName = modelName
        self.type = result_type
        self.device = torch.device('cpu')

        if self.modelName == "FCN":
            # FCN setting
            self.model_fcn = torchfcn.models.FCN8s(n_class=2)
            self.model_fcn.eval()
            self.model_data = torch.load(f'./segmentationModel/FCN_model_best_{seg_type}.pth.tar',map_location=self.device) #model file 위치
            try:
                self.model_fcn.load_state_dict(self.model_data)
            except Exception:
                self.model_fcn.load_state_dict(self.model_data['model_state_dict'])
            self.mean_bgr = np.array([104.00698793, 116.66876762, 122.67891434])
            
        elif self.modelName == "DeepLab" :
            # DeepLab setting
            self.model_deep = DeepLab(num_classes = 2,
                        backbone = 'xception',
                        output_stride = 16,
                        sync_bn = False,
                        freeze_bn = False)
            checkpoint_deeplab = torch.load('./segmentationModel/DeepLab_model_best.pth.tar' ,map_location=self.device)
            self.model_deep.load_state_dict(checkpoint_deeplab['state_dict'])
            self.model_deep.eval()
            self.composed_transforms = transforms.Compose([
                transforms.ToTensor(),
                transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))])
            
        elif self.modelName == "U_Net": 
            # U-Net setting
            self.model_uNet = tf.keras.models.load_model('./segmentationModel/UNet_0823_newdataset_pat20.h5')
            pass
        else :
            print("SegmentationCam.py __init__ 에서 어느 모델도 초기화 X") 
            pass
        print("초기화 완료")


    def predictColoring(self, frame, pred_model):
        try : 
            score = np.bincount(pred_model.flatten().tolist())[1]
            score = round((score/(pred_model.shape[0]*pred_model.shape[1]))*100, 4)
        except :
            score = 0
            
        color = (255,0,0)  
        
        
        if self.type == "mask":
            color = (255,255,255)
            frame=0 
            frame = np.zeros(shape=(pred_model.shape[0],pred_model.shape[1],3))
          
        cnt=0
        for i in range(pred_model.shape[0]):
            for j in range(pred_model.shape[1]):
                if pred_model[i][j] == 1:
                    frame[i][j] = color
                    cnt+=1

        return {"frame":frame, "score":score}
    
    def model(self, frame):
        if self.modelName == "FCN":
            #openCV Image to PIL Image
            image = cv2.cvtColor(frame,cv2.COLOR_BGR2RGB)
            img = Image.fromarray(image).convert('RGB')

            #FCN image
            img = np.array(img, dtype=np.uint8)[:, :, ::-1] # RGB -> BGR
            img = img.astype(np.float64)
            img -= self.mean_bgr
            img = img.transpose(2, 0, 1)
            img = torch.from_numpy(img).float().unsqueeze(0)

            #FCN
            pred_fcn = self.model_fcn(img)
            pred_fcn = pred_fcn.data.max(1)[1].cpu().numpy()[0]

            result = self.predictColoring(frame, pred_fcn)
            return result
        
        elif self.modelName == "DeepLab" :

            image = cv2.cvtColor(frame,cv2.COLOR_BGR2RGB)
            image = Image.fromarray(image).convert('RGB')
            image = self.composed_transforms(image)
            image = image.unsqueeze(0)

            with torch.no_grad():
                pred_deep = self.model_deep(image)
            
            pred_deep = pred_deep.data.cpu().numpy()
            pred_deep = np.argmax(pred_deep, axis=1)
            pred_deep = pred_deep[0]

            result = self.predictColoring(frame, pred_deep)
            return result
        
        elif self.modelName == "U_Net" : 
            #u-net iamge
            h, w, _ = frame.shape

            frame_unet = cv2.resize(frame,(112,112))
            image_unet = frame_unet
            image_unet = np.reshape(image_unet,(112,112,-1))
            image_unet = np.array([image_unet])/255
            pred_unet = self.model_uNet.predict(image_unet)
            pred_unet = pred_unet[0]

            _, pred_unet = cv2.threshold(np.array(pred_unet*255,dtype='uint8'),0,1,cv2.THRESH_BINARY)
            
            result= self.predictColoring(frame_unet, pred_unet)
            result['frame'] = cv2.resize(result['frame'],(w, h),  interpolation=cv2.INTER_CUBIC)
            return result
        else :
            pass

### SegmentationModels Class 실행

In [3]:
import cv2
import numpy as np

# FCN
from PIL import Image
import torch
import torchfcn

# U-Net
import tensorflow as tf

# DeepLab
from segmentationModel.DeepLabModeling.deeplab import *
from torchvision import transforms

2023-06-30 02:26:04.728024: I tensorflow/core/util/port.cc:110] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2023-06-30 02:26:04.802632: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2023-06-30 02:26:05.989863: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


In [4]:
import pandas as pd 
df = pd.DataFrame(columns=["name","score"])
df

,name,score


### result
#### ver. color (원본과 같이 출력)

In [ ]:
# import os 
# import matplotlib.pyplot as plt
# from tqdm import tqdm

# model_list = ["FCN","DeepLab","U_Net"]


# for model in model_list:
#     print(model)
#     Seg = SegmentationModels(model,"color")

#     !rm -rf ./segmentationModel/results/{dataset_name}/{model}/
#     !mkdir ./segmentationModel/results/{dataset_name}/
#     !mkdir ./segmentationModel/results/{dataset_name}/{model}/
#     save_path = f"./segmentationModel/results/{dataset_name}/{model}/"



#     for img_name in tqdm(imgs):

#         img=cv2.imread(origin_path+img_name, cv2.IMREAD_COLOR)
# #         img = cv2.resize(img, (133,133)) 

#         img = cv2.cvtColor(img,cv2.COLOR_BGR2RGB)
#     #     plt.imshow(img)
#     #     plt.show()

#         seg = Seg.model(img.copy())
#         plt.imshow(seg["frame"])
#         plt.show()


#         add_img = cv2.cvtColor(cv2.hconcat([seg["frame"],img]),cv2.COLOR_BGR2RGB)
#     #     plt.imshow(add_img)
#     #     plt.show()

#         cv2.imwrite(save_path+img_name+f"_{seg['score']}.jpg",add_img)
#         df= df.append({"name":img_name, "score":seg['score']}, ignore_index=True)
        
#         break
#     df.to_csv(f"{save_path}segScore.csv")
    


#### ver. mask 

In [5]:
import os 
import matplotlib.pyplot as plt
from tqdm import tqdm

model_list = ["FCN"] #,"DeepLab","U_Net"

seg1 = 0

for model in model_list:
    print(model)
    Seg = SegmentationModels(model,"mask")

    !rm -rf ./segmentationModel/results/{dataset_name}_mask/{model}/
    !mkdir ./segmentationModel/results/{dataset_name}_mask/
    !mkdir ./segmentationModel/results/{dataset_name}_mask/{model}/
    save_path = f"./segmentationModel/results/{dataset_name}_mask/{model}/"



    for img_name in tqdm(imgs):

        img=cv2.imread(origin_path+img_name, cv2.IMREAD_COLOR)
#         plt.imshow(cv2.cvtColor(img,cv2.COLOR_BGR2RGB))
#         plt.show()
#         img = cv2.resize(img, (133,133)) 

#         img = cv2.cvtColor(img,cv2.COLOR_BGR2RGB)
    #     plt.imshow(img)
    #     plt.show()

        seg = Seg.model(img.copy())
        seg['frame']= seg['frame'].astype('uint8')
#         plt.imshow(seg["frame"])
#         plt.show()
#         print(seg["frame"])
#         print(seg["frame"].shape)
#         break
        

        cv2.imwrite(save_path+img_name+f"_{seg['score']}.jpg", seg['frame'])
        df= df.append({"name":img_name, "score":seg['score']}, ignore_index=True)

    df.to_csv(f"{save_path}segScore.csv")
    


FCN
SegmentationCam.py __init__ 들어옴
초기화 완료
mkdir: cannot create directory ‘./segmentationModel/results/20230605_ch1_crop_mask/’: File exists


100% 623/623 [23:42<00:00,  2.28s/it]


In [ ]:
df

In [ ]:
seg1

----


MultiModal_SolarPowerGenerationForecast-AAA24-forComparison.ipynb 를 위한
segmentation 결과 csv 만들기

In [53]:
import numpy as np
import os
from collections import Counter
import pandas as pd
from tqdm import tqdm

folder_path = '/app/MultiModal/dataset/seg_result_20240723/' 
save_path='/app/MultiModal/dataset/seg_result_csv_20240723/'

os.makedirs(save_path, exist_ok=True)

seg_models = os.listdir(folder_path)
seg_models.remove('.ipynb_checkpoints')

for seg_model in seg_models:
    
    df = pd.DataFrame(columns=['name', 'panel number', 'panel', 'bird dropping', 'pine pollen', 'plastic bag', 'sand', 'shadow'])
    
    for file_name in tqdm(os.listdir(os.path.join(folder_path,seg_model))):
        
        file_path =os.path.join(folder_path,seg_model,file_name)
        
        data = np.load(file_path)
        data = np.argmax(data, axis=-1)
        
        row = [0.0]*6
        
        # 전체 배열을 1차원으로 변환
        flattened_array = data.flatten()

        # 값의 빈도수 계산
        value_counts = Counter(flattened_array)

        # 결과 출력
        for value, count in value_counts.items():
            row[value] = count/len(flattened_array)*100

        panel_num = file_name[5]
#         date = file_name[19:-4]

        row=[file_name, panel_num]+row
        
        new_row = pd.DataFrame([row], columns=df.columns)
        df = df.append(new_row, ignore_index=True)
        
    df.to_csv(f'{save_path}{seg_model}.csv')
    df

100% 1707/1707 [00:25<00:00, 66.15it/s]
100% 1707/1707 [00:29<00:00, 58.83it/s]
100% 1707/1707 [00:27<00:00, 62.94it/s]
100% 1707/1707 [00:28<00:00, 58.99it/s]
100% 1707/1707 [00:25<00:00, 65.85it/s]
